# Experiment: S6E8 智能手机成瘾 EDA

**目标**：弄清特征分布、缺失模式与 `addicted_label` 的关系，指导特征工程与建模。

**成功标准**：识别强信号特征、确认 train/test 分布一致性、输出可落地的建模建议。

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)
ROOT = Path(r"e:\py_project\playground-series\playground-series-s6e8")
sns.set_theme(style="whitegrid", context="notebook")
ROOT

## Plan

- 检查形状、目标分布、缺失率
- 数值特征与目标的相关 / 分箱成瘾率
- 类别特征目标率
- train vs test 分布对比

In [ ]:
train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")
TARGET = "addicted_label"
NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

{
    "train_shape": train.shape,
    "test_shape": test.shape,
    "pos_rate": float(train[TARGET].mean()),
    "missing_train_pct": (train.isna().mean() * 100).round(2).sort_values(ascending=False).to_dict(),
}

In [ ]:
corr = train[NUM_COLS + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("与目标相关性:\n", corr.round(4))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, ["daily_screen_time_hours", "weekend_screen_time", "social_media_hours"]):
    tmp = train[[col, TARGET]].dropna()
    tmp["bin"] = pd.qcut(tmp[col], 5, duplicates="drop")
    rate = tmp.groupby("bin", observed=True)[TARGET].mean()
    rate.plot(kind="bar", ax=ax, color="#3b82f6")
    ax.set_title(col)
    ax.set_ylabel("addiction rate")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Results

- 强信号：`daily_screen_time_hours` / `weekend_screen_time` / `social_media_hours`
- 类别特征单独区分力弱；缺失率 train/test 接近
- **决策**：GBDT 三模型 + OOF 权重融合，提交概率

In [ ]:
result = {
    "top_corr": corr.head(5).round(4).to_dict(),
    "decision": "树模型 + 屏幕衍生特征 + 概率融合",
}
result